# Squareprompt Capital baseline

Each stage receives a copy of completed historical bars and a state dictionary. Return the state. No future prices, filesystem, network, or holdout access. The trusted evaluator controls chronological replay, fills, fees, funding, exposure caps and scoring. External point-in-time sources arrive in state["sources"].

In [ ]:
def stage_sources(history, state):
    # Managed sources are already filtered by publication time.
    return state


In [ ]:
def stage_quality(history, state):
    if history["close"].isna().any():
        raise ValueError("Missing close prices")
    return state


In [ ]:
def stage_storage(history, state):
    state["as_of"] = int(history["available_at"].iloc[-1])
    return state


In [ ]:
def stage_state(history, state):
    state["volatility"] = float(history["close"].pct_change().tail(24).std())
    return state


In [ ]:
def stage_models(history, state):
    close = history["close"]
    window = int(state.get("training_window", 24))
    state["forecast"] = float(close.iloc[-1] / close.iloc[-window] - 1)
    return state


In [ ]:
def stage_signals(history, state):
    forecast = state.get("forecast", 0.0)
    threshold = 0.002
    state["target"] = 1.0 if forecast > threshold else (-1.0 if forecast < -threshold else 0.0)
    return state


In [ ]:
def stage_sizing(history, state):
    state["target"] *= 0.35
    return state


In [ ]:
def stage_exposure(history, state):
    state["target"] = max(-0.8, min(0.8, state["target"]))
    return state


In [ ]:
def stage_risk(history, state):
    if state["drawdown"] > 0.15:
        state["target"] = 0.0
    return state


In [ ]:
def stage_retraining(history, state):
    state["training_window"] = 24
    return state


In [ ]:
def stage_drift(history, state):
    if state.get("volatility", 0) > 0.08:
        state["target"] *= 0.5
    return state


In [ ]:
def stage_registry(history, state):
    state["pipeline_version"] = "baseline-v1"
    return state
